# Offline vs Online Hybrid CV-RAG Results

This notebook documents the verified hybrid run after provisioning Azure AI Search and a same-resource-group `gpt-5.4-mini` Foundry deployment. It compares the fully offline seed pack with the online enriched corpus, then shows how online-only results are retained in a local SQLite delta store for later disconnected retrieval.

## Runtime flow verified in the POC

```mermaid
flowchart LR
  Query[Field query + optional site photo] --> OfflineEmbed[Local CLIP text/image embedding]
  OfflineEmbed --> OfflineSQLite[Offline SQLite seed pack: 6 cases]
  OfflineSQLite --> OfflineAnswer[Offline result via Phi-4-mini/template answer]
  Query -->|network available| Search[Azure AI Search: 12 GPT-enriched cases]
  Search --> OnlineAnswer[Online result with richer checklist + escalation]
  Search --> Sync[Select online-only high-value cases]
  Sync --> DeltaSQLite[Offline SQLite delta pack]
  DeltaSQLite --> FutureOffline[Future offline search finds synced online cases]
```


## Provisioned online components

| Component | POC resource |
| --- | --- |
| Resource group | `rg-hybrid-rag-slm-poc` |
| GPU VM | `vm-cvrag-a10-poc`, `Standard_NV6ads_A10_v5`, South Central US |
| Azure AI Search | `srch-hybrid-rag-887070`, index `construction-incidents-online`, 12 documents |
| Foundry / AI Services | `aif-hybrid-rag-338698`, custom endpoint enabled for Entra token auth |
| Text model | `gpt-5.4-mini`, deployment `gpt-5-4-mini-enrich`, GlobalStandard capacity 10 |
| Image model | `MAI-Image-2.5-Flash` was not newly deployed in South Central US because quota is fully used: current 2, limit 2. The notebook keeps generated image captions/prompts and uses committed local synthetic images until image quota is available. |


## What got indexed online

The online index was rebuilt from `assets/online_comparison/gpt54mini_enriched_incidents.json`, generated by the `gpt-5.4-mini` deployment. Each record includes a title, severity, visual clues, image caption, observation, root-cause hypothesis, action checklist, escalation rule, and offline-cache reason. The index stores a 512-dimensional CLIP text embedding in `content_vector` and searchable text fields for hybrid keyword/vector retrieval.

| ID | Title | Severity | Source scope |
| --- | --- | --- | --- |
| INC-001 | Basement wall water ingress at cold joint | high | offline_seed_enriched |
| INC-002 | Concrete column honeycombing at lower lift | medium | offline_seed_enriched |
| INC-003 | Rebar congestion at beam-column joint | high | offline_seed_enriched |
| INC-004 | MEP duct clashes with ceiling coordination zone | medium | offline_seed_enriched |
| INC-005 | Unsafe open edge adjacent to scaffold access | critical | offline_seed_enriched |
| INC-006 | Crack observed near lift core wall opening | high | offline_seed_enriched |
| ONL-007 | Water ingress detected near temporary electrical riser | critical | online_enriched_only |
| ONL-008 | Falling object exclusion-zone breach below overhead works | high | online_enriched_only |
| ONL-009 | Concrete spalling at slab edge after formwork striking | medium | online_enriched_only |
| ONL-010 | Temporary works prop deformation observed under load | critical | online_enriched_only |
| ONL-011 | Confined-space gas detector alarm during drainage inspection | critical | online_enriched_only |
| ONL-012 | Mobile crane lift conducted near overhead service line | critical | online_enriched_only |

In [1]:
import json
from pathlib import Path

summary = json.loads(Path("assets/online_comparison/offline_online_summary.json").read_text())
print(f"AI Search documents indexed: {summary['azure_ai_search']['documents_indexed']}")
print(f"Comparison queries: {summary['vm_run']['comparison_queries']}")
print(f"Online-only cases cached into offline delta store: {summary['vm_run']['offline_delta_count']}")


AI Search documents indexed: 12
Comparison queries: 4
Online-only cases cached into offline delta store: 4


## Offline vs online retrieval result

| Scenario | Offline top | Online top | Synced offline top | Interpretation |
| --- | --- | --- | --- | --- |
| Known offline concrete defect | INC-002 | INC-002 | n/a | The local seed pack is sufficient; no online-only delta is required. |
| Water ingress upgraded by electrical risk | INC-001 | ONL-007 | ONL-007 | Online retrieval upgrades a generic waterproofing case into a critical electrical-safety case and caches it offline. |
| Temporary works case missing offline | INC-005 | ONL-010 | ONL-010 | The offline seed pack has no temporary-works case; AI Search finds the enriched online case and the delta store retains it. |
| Confined space safety case missing offline | INC-005 | ONL-011 | ONL-011 | Online retrieval fills a confined-space safety gap and preserves the case for future disconnected searches. |

## Interpretation

The fully offline pack is sufficient for common, preloaded defects such as concrete honeycombing. The online path becomes valuable when the field condition includes a risk combination or discipline that the local pack has not cached yet, such as water plus temporary electrical equipment, temporary works deformation, or confined-space atmospheric alarms.

After an online-only case is retrieved, the demo writes the selected record and vector into `SyncedCaseStore` (`offline_delta.sqlite`). The follow-up offline search then finds `ONL-007`, `ONL-010`, and `ONL-011` without calling Azure AI Search.

## Example online-enriched actions

### ONL-007 - Water ingress detected near temporary electrical riser

**Why online helps:** Online retrieval upgrades a generic waterproofing case into a critical electrical-safety case and caches it offline.

**Root-cause hypothesis:** Likely external water infiltration, failed weather protection, or inadequate segregation between wet work and temporary power distribution.

1. Isolate the affected temporary electrical circuit and verify de-energization
1. Restrict access to the wet zone and place warning signage
1. Trace the water source and implement temporary diversion or containment
1. Inspect all nearby plugs, junctions, and cable insulation for damage
1. Only re-energize after a competent electrical inspection and dryness verification

**Escalation:** Escalate immediately to electrical supervision and site management if any energized equipment has been exposed to moisture.

### ONL-010 - Temporary works prop deformation observed under load

**Why online helps:** The offline seed pack has no temporary-works case; AI Search finds the enriched online case and the delta store retains it.

**Root-cause hypothesis:** Possible overloading, incorrect prop spacing, inadequate bracing, settlement at the base, or unauthorized removal of adjacent supports.

1. Immediately stop loading and clear the area beneath the supported element
1. Install secondary support or stabilization only under approved temporary works design
1. Inspect all adjacent props, braces, and base conditions for distress
1. Compare the installation against the approved temporary works sequence and load rating
1. Do not resume work until the temporary works engineer signs off the condition

**Escalation:** Escalate immediately to the temporary works engineer and site manager if any prop is deformed, displaced, or audibly creaking under load.

### ONL-011 - Confined-space gas detector alarm during drainage inspection

**Why online helps:** Online retrieval fills a confined-space safety gap and preserves the case for future disconnected searches.

**Root-cause hypothesis:** Possible oxygen deficiency, buildup of toxic gas, or inadequate purge/ventilation before inspection.

1. Abort entry and remove personnel from the confined space area
1. Maintain isolation and prevent re-entry until atmospheric testing is cleared
1. Verify detector calibration, battery status, and sensor function
1. Introduce forced ventilation and retest atmosphere from outside the space
1. Resume only under an approved confined-space permit with rescue provisions in place

**Escalation:** Escalate immediately to confined-space supervision and emergency response if any person has entered the space after the alarm or feels unwell.

## How to rerun

```bash
python scripts/generate_enriched_incidents_with_foundry.py   --endpoint "$AZURE_OPENAI_ENDPOINT"   --deployment "$AZURE_OPENAI_DEPLOYMENT"   --output notebooks/assets/online_comparison/gpt54mini_enriched_incidents.json

python scripts/build_online_index.py   --endpoint "$SEARCH_ENDPOINT"   --key "$SEARCH_KEY"   --index construction-incidents-online   --device cuda   --incidents-json notebooks/assets/online_comparison/gpt54mini_enriched_incidents.json

python scripts/run_offline_online_comparison.py   --endpoint "$SEARCH_ENDPOINT"   --key "$SEARCH_KEY"   --index construction-incidents-online   --workspace data/hybrid-comparison   --device cuda   --incidents-json notebooks/assets/online_comparison/gpt54mini_enriched_incidents.json
```

For the full offline path, keep using `notebooks/offline_cv_rag_results.ipynb`; it now embeds image previews directly so GitHub can render the examples even when relative image links are not expanded.